# 第2章 · 2.1 网络硬件与传输介质 (Network Hardware & Transmission Media)
> Cambridge International AS & A Level Computer Science (9618)

---

本 notebook 将从零开始介绍：
1. 网络接口卡 (NIC / WNIC)
2. 交换机 (Switch) 与集线器 (Hub)
3. 路由器 (Router)
4. 无线接入点 (WAP)、网桥 (Bridge)、中继器 (Repeater)
5. 服务器类型
6. 传输介质：铜缆、光纤、无线
7. 有线 vs 无线对比
8. 练习题

## 1. 网络接口卡 NIC (Network Interface Card)

### 1.1 NIC 是什么？

**NIC（网络接口卡）** 是让你的电脑能够"说网络语言"的硬件。

> **类比**：NIC 就像你嘴巴和耳朵的组合——让你能"说出"数据（发送）和"听到"数据（接收）。没有 NIC，你的电脑就是个哑巴，无法和其他电脑交流。

### 1.2 NIC 做了什么？

1. **数据转换**：把电脑内部的数字数据转换成可以在网线/无线中传输的信号
   - 有线 NIC：转换为电信号
   - 无线 WNIC：转换为无线电波
2. **MAC 地址**：每块 NIC 都有一个全球唯一的 **MAC 地址 (Media Access Control Address)**
   - 格式：`00:1A:2B:3C:4D:5E`（6 组十六进制数）
   - 就像网卡的"身份证号"——全世界没有两块网卡的 MAC 地址相同
3. **数据的打包和拆包**：把数据封装成适合网络传输的"数据帧 (Frame)"

### 1.3 NIC vs WNIC

| 特性 | NIC (有线) | WNIC (无线) |
|------|-----------|------------|
| 连接方式 | 网线 (Ethernet cable) | WiFi 无线信号 |
| 速度 | 更快更稳定 | 受干扰影响 |
| 移动性 | 差（被线束缚） | 好（可以自由移动） |
| 安全性 | 高（需要物理连接） | 较低（信号可以被截获） |

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 SimHei 字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DengXian', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('SimHei')
if 'SimHei' in test_font or 'simhei' in test_font.lower():
    print('✅ 中文字体 SimHei 已加载')
else:
    print(f'⚠️ SimHei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = 'C:/Windows/Fonts/simhei.ttf'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['SimHei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 SimHei 字体文件')

# 查看你的电脑的网络接口信息（模拟数据展示 MAC 地址格式）
import random

def generate_mac():
    # 生成一个模拟的 MAC 地址
    return ':'.join([f'{random.randint(0, 255):02X}' for _ in range(6)])

print("=" * 55)
print("  模拟：网络接口卡 (NIC) 信息")
print("=" * 55)

interfaces = [
    ("Ethernet (有线 NIC)", generate_mac(), "1 Gbps", "Connected"),
    ("Wi-Fi (无线 WNIC)", generate_mac(), "867 Mbps", "Connected"),
    ("Bluetooth", generate_mac(), "3 Mbps", "Disconnected"),
]

for name, mac, speed, status in interfaces:
    print(f"\n  Interface: {name}")
    print(f"  MAC Address: {mac}")
    print(f"  Speed: {speed}")
    print(f"  Status: {status}")

print("\n" + "=" * 55)
print("MAC 地址是 48 bits (6 bytes)，用十六进制表示")
print("前 3 bytes (OUI) 标识制造商，后 3 bytes 是设备序列号")
print("例如: 00:1A:2B 可能是 Cisco 的设备")


## 2. 交换机 (Switch) vs 集线器 (Hub)

### 2.1 Hub（集线器）—— 简单但"笨"

**Hub** 是最简单的网络连接设备。它收到数据后，会把数据 **广播给所有端口**。

> **类比**：Hub 就像一个 **没有指向性的喇叭**——不管你想和谁说话，所有人都能听到。

**问题：**
- 浪费带宽（所有人都收到数据，但只有一个人需要）
- 安全性差（所有数据对所有设备可见）
- 容易发生冲突（多台设备同时发送数据会碰撞）

### 2.2 Switch（交换机）—— 智能转发

**Switch** 比 Hub 聪明得多。它会学习每个端口连接的设备的 **MAC 地址**，然后只把数据发送到 **正确的端口**。

> **类比**：Switch 就像一个 **邮局分拣员**——看到地址后，把信准确地投递到收件人，而不是给所有人都发一份。

**Switch 的工作原理：**
1. Switch 内部维护一个 **MAC 地址表**（MAC Address Table）
2. 当收到数据帧时，查看目标 MAC 地址
3. 在 MAC 地址表中查找对应的端口
4. 只把数据转发到那个端口
5. 如果不知道目标在哪个端口，就广播到所有端口（学习过程）

### 2.3 对比

| 特性 | Hub | Switch |
|------|-----|--------|
| 智能程度 | 无脑广播 | 智能转发 |
| 数据发送方式 | 发给所有端口 | 只发给目标端口 |
| 带宽利用 | 所有设备共享带宽 | 每个端口独享带宽 |
| 碰撞 | 频繁碰撞 | 几乎无碰撞 |
| 安全性 | 低 | 较高 |
| 现在还用吗？ | 几乎淘汰 | 广泛使用 |

In [ ]:
# 模拟：Hub vs Switch 的数据转发行为
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Hub behavior ──
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Hub: Broadcast to ALL', fontsize=14, fontweight='bold', color='#E53935')
ax.set_aspect('equal')
ax.axis('off')

# Hub center
hub = plt.Circle((5, 5), 0.6, color='#E53935', ec='#B71C1C', lw=2)
ax.add_patch(hub)
ax.text(5, 5, 'Hub', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Devices
angles_h = np.linspace(0, 2*np.pi, 5, endpoint=False) - np.pi/2
for i, a in enumerate(angles_h):
    x = 5 + 3.5 * np.cos(a)
    y = 5 + 3.5 * np.sin(a)
    color = '#4CAF50' if i == 0 else ('#2196F3' if i == 2 else '#9E9E9E')
    label = 'Sender' if i == 0 else ('Target' if i == 2 else f'PC{i+1}')
    r = mpatches.FancyBboxPatch((x-0.6, y-0.35), 1.2, 0.7, boxstyle="round,pad=0.1",
                                  facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(r)
    ax.text(x, y, label, ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    # ALL get data (red arrows)
    ax.annotate('', xy=(x, y), xytext=(5, 5),
                arrowprops=dict(arrowstyle='->', color='red', lw=2, alpha=0.6))

ax.text(5, 0.3, 'Data sent to ALL ports!\n(wasteful & insecure)', ha='center',
        fontsize=10, color='#E53935', style='italic')

# ── Switch behavior ──
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_title('Switch: Forward to TARGET only', fontsize=14, fontweight='bold', color='#4CAF50')
ax2.set_aspect('equal')
ax2.axis('off')

# Switch center
sw = mpatches.FancyBboxPatch((4.25, 4.55), 1.5, 0.9, boxstyle="round,pad=0.15",
                               facecolor='#4CAF50', edgecolor='#2E7D32', lw=2)
ax2.add_patch(sw)
ax2.text(5, 5, 'Switch', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

for i, a in enumerate(angles_h):
    x = 5 + 3.5 * np.cos(a)
    y = 5 + 3.5 * np.sin(a)
    is_sender = (i == 0)
    is_target = (i == 2)
    color = '#4CAF50' if is_sender else ('#2196F3' if is_target else '#BDBDBD')
    label = 'Sender' if is_sender else ('Target' if is_target else f'PC{i+1}')
    r = mpatches.FancyBboxPatch((x-0.6, y-0.35), 1.2, 0.7, boxstyle="round,pad=0.1",
                                  facecolor=color, edgecolor='black', lw=1.5)
    ax2.add_patch(r)
    ax2.text(x, y, label, ha='center', va='center', fontsize=8, color='white', fontweight='bold')

    if is_sender:
        ax2.annotate('', xy=(5, 5), xytext=(x, y),
                     arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
    elif is_target:
        ax2.annotate('', xy=(x, y), xytext=(5, 5),
                     arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
    else:
        ax2.plot([5, x], [5, y], '-', lw=1, color='#E0E0E0', alpha=0.5)

ax2.text(5, 0.3, 'Data sent ONLY to the target port!\n(efficient & more secure)', ha='center',
         fontsize=10, color='#4CAF50', style='italic')

plt.tight_layout()
plt.savefig('hub_vs_switch.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. 路由器 (Router)

### 3.1 Router 做什么？

**Router（路由器）** 连接 **不同的网络**，并决定数据包应该走哪条路径。

> **类比**：Router 就像 **十字路口的交通警察**——它知道每个方向通向哪里，指挥数据包走最好的路线到达目的地。

### 3.2 Router vs Switch — 关键区别

| 特性 | Switch | Router |
|------|--------|--------|
| 工作层级 | 数据链路层 (Layer 2) | 网络层 (Layer 3) |
| 使用的地址 | MAC 地址 | IP 地址 |
| 连接的范围 | 同一个网络内的设备 | 不同网络之间 |
| 广播域 | 一个 | 分隔广播域 |

### 3.3 Router 的工作原理

1. 收到一个 **数据包 (Packet)**
2. 查看数据包的 **目标 IP 地址**
3. 查询内部的 **路由表 (Routing Table)**
4. 决定下一跳（next hop）——把数据包转发到哪个网络接口
5. 如果有多条路径，选择最优的一条

### 3.4 为什么需要 Router？

想象一下：
- 你家的 WiFi 网络是一个 LAN
- 你要访问百度（在互联网 WAN 上）
- 你的家用路由器就是连接你家 LAN 和 ISP（互联网服务提供商）网络的桥梁

没有 Router，你只能和同一个 LAN 里的设备通信，无法访问互联网！

In [ ]:
# 可视化：Router 连接不同网络
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(0, 20)
ax.set_ylim(0, 10)
ax.set_title('Router: Connecting Different Networks', fontsize=14, fontweight='bold')
ax.axis('off')

# Network A (LAN)
net_a = mpatches.FancyBboxPatch((0.5, 1), 5.5, 8, boxstyle="round,pad=0.3",
                                  facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(net_a)
ax.text(3.25, 8.5, 'Network A (192.168.1.x)', ha='center', fontsize=11,
        fontweight='bold', color='#1565C0')

# Switch A
sw_a = mpatches.FancyBboxPatch((2.5, 5.2), 1.5, 0.7, boxstyle="round,pad=0.1",
                                 facecolor='#FF7043', edgecolor='#D84315', lw=2)
ax.add_patch(sw_a)
ax.text(3.25, 5.55, 'Switch', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# PCs in Network A
for i, (x, y, ip) in enumerate([(1.5, 3, '.1'), (3.25, 3, '.2'), (5, 3, '.3')]):
    r = mpatches.FancyBboxPatch((x-0.55, y-0.35), 1.1, 0.7, boxstyle="round,pad=0.1",
                                  facecolor='#42A5F5', edgecolor='#1565C0', lw=1.5)
    ax.add_patch(r)
    ax.text(x, y, f'PC\n{ip}', ha='center', va='center', fontsize=7, color='white')
    ax.plot([x, 3.25], [y+0.35, 5.2], 'b-', lw=1, alpha=0.5)

# Router
router = mpatches.FancyBboxPatch((8.5, 4.2), 3, 1.6, boxstyle="round,pad=0.2",
                                   facecolor='#AB47BC', edgecolor='#6A1B9A', lw=3)
ax.add_patch(router)
ax.text(10, 5.3, 'Router', ha='center', fontsize=13, color='white', fontweight='bold')
ax.text(10, 4.6, 'Routing Table', ha='center', fontsize=9, color='#E1BEE7')

# Connection A -> Router
ax.annotate('', xy=(8.5, 5.0), xytext=(6, 5.55),
            arrowprops=dict(arrowstyle='<->', color='#1565C0', lw=2.5))

# Network B (LAN)
net_b = mpatches.FancyBboxPatch((14, 1), 5.5, 8, boxstyle="round,pad=0.3",
                                  facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(net_b)
ax.text(16.75, 8.5, 'Network B (10.0.0.x)', ha='center', fontsize=11,
        fontweight='bold', color='#2E7D32')

# Switch B
sw_b = mpatches.FancyBboxPatch((16, 5.2), 1.5, 0.7, boxstyle="round,pad=0.1",
                                 facecolor='#FF7043', edgecolor='#D84315', lw=2)
ax.add_patch(sw_b)
ax.text(16.75, 5.55, 'Switch', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

for i, (x, y, ip) in enumerate([(15.25, 3, '.1'), (16.75, 3, '.2'), (18.25, 3, '.3')]):
    r = mpatches.FancyBboxPatch((x-0.55, y-0.35), 1.1, 0.7, boxstyle="round,pad=0.1",
                                  facecolor='#66BB6A', edgecolor='#2E7D32', lw=1.5)
    ax.add_patch(r)
    ax.text(x, y, f'PC\n{ip}', ha='center', va='center', fontsize=7, color='white')
    ax.plot([x, 16.75], [y+0.35, 5.2], 'g-', lw=1, alpha=0.5)

# Connection Router -> B
ax.annotate('', xy=(14, 5.55), xytext=(11.5, 5.0),
            arrowprops=dict(arrowstyle='<->', color='#2E7D32', lw=2.5))

# Internet cloud
cloud = mpatches.FancyBboxPatch((7.5, 7.5), 5, 1.5, boxstyle="round,pad=0.3",
                                  facecolor='#EEEEEE', edgecolor='#616161', lw=2)
ax.add_patch(cloud)
ax.text(10, 8.25, 'Internet', ha='center', fontsize=12, fontweight='bold', color='#424242')
ax.annotate('', xy=(10, 7.5), xytext=(10, 5.8),
            arrowprops=dict(arrowstyle='<->', color='#616161', lw=2))

plt.tight_layout()
plt.savefig('router_networks.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. 其他网络设备

### 4.1 WAP (Wireless Access Point) — 无线接入点

**WAP** 让无线设备（手机、笔记本）能够连接到有线网络。

> **类比**：WAP 就像一个 **翻译官**——它把无线信号翻译成有线信号，让两种"语言"的设备能够交流。

**工作方式：**
1. 无线设备通过 WiFi 连接到 WAP
2. WAP 连接到有线网络的 Switch
3. WAP 在有线和无线之间转换数据

**注意**：家里的"WiFi路由器"其实是三合一设备：Router + Switch + WAP

### 4.2 Bridge（网桥）

**Bridge** 连接两个 **相同类型** 的网络段，减少网络拥塞。

> **类比**：Bridge 就像一座真正的 **桥**——连接河两岸（两个网段），让数据可以通过。

**工作方式：**
- 学习每一侧的 MAC 地址
- 只有当数据的目标在"桥的另一边"时才转发
- 减少了不必要的流量

### 4.3 Repeater（中继器）

**Repeater** 放大和再生网络信号，延长传输距离。

> **类比**：Repeater 就像 **接力赛跑的接棒人**——信号跑累了（衰减了），Repeater 接过来重新跑。

**为什么需要 Repeater？**
- 信号在传输过程中会 **衰减 (Attenuation)**——越远越弱
- 超过一定距离，信号就弱到无法被正确接收
- Repeater 在信号变弱之前接收它、放大它、然后重新发送

### 4.4 常见服务器类型

| 服务器类型 | 功能 | 例子 |
|-----------|------|------|
| File Server (文件服务器) | 集中存储和管理文件 | 学校共享文件夹 |
| Print Server (打印服务器) | 管理网络打印机 | 办公室打印队列 |
| Web Server (Web服务器) | 托管网站 | Apache, Nginx |
| Mail Server (邮件服务器) | 处理电子邮件 | Exchange, Gmail |
| DNS Server | 域名解析 | 将域名转为IP |
| DHCP Server | 自动分配IP地址 | 路由器内置功能 |
| Proxy Server (代理服务器) | 代替客户端访问网络 | 学校网络过滤 |

In [ ]:
# 可视化：网络设备层级关系
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.set_title('Network Devices: What Each One Does', fontsize=14, fontweight='bold')
ax.axis('off')

devices = [
    (2, 8, 'Repeater', '#9E9E9E', 'Layer 1 (Physical)', 'Amplifies signals\nExtends distance'),
    (7, 8, 'Hub', '#BDBDBD', 'Layer 1 (Physical)', 'Broadcasts to all\n(Obsolete)'),
    (2, 5.5, 'Bridge', '#FF9800', 'Layer 2 (Data Link)', 'Connects 2 segments\nFilters by MAC'),
    (7, 5.5, 'Switch', '#FF7043', 'Layer 2 (Data Link)', 'Smart forwarding\nMAC address table'),
    (12, 5.5, 'WAP', '#26C6DA', 'Layer 2 (Data Link)', 'Wireless to wired\nWiFi access'),
    (7, 2.5, 'Router', '#AB47BC', 'Layer 3 (Network)', 'Connects networks\nIP routing'),
]

for x, y, name, color, layer, desc in devices:
    rect = mpatches.FancyBboxPatch((x-1.5, y-0.9), 3, 1.8, boxstyle="round,pad=0.2",
                                     facecolor=color, edgecolor='black', linewidth=2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x, y+0.3, name, ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    ax.text(x, y-0.3, desc, ha='center', va='center', fontsize=7, color='white')
    ax.text(x, y+0.9, layer, ha='center', fontsize=8, color='#333', style='italic',
            bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.7))

# Layer labels on the left
for y_pos, layer_name in [(8, 'Physical Layer'), (5.5, 'Data Link Layer'), (2.5, 'Network Layer')]:
    ax.text(-0.2, y_pos, layer_name, fontsize=10, fontweight='bold', color='#555',
            rotation=0, va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#F5F5F5', edgecolor='#999'))

# Arrows showing "smarter"
ax.annotate('', xy=(7, 1.2), xytext=(7, 9.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=2, ls='--'))
ax.text(13.5, 8, 'Simpler', fontsize=11, color='gray', ha='center')
ax.text(13.5, 2.5, 'Smarter', fontsize=11, color='#AB47BC', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('network_devices.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. 传输介质 (Transmission Media)

传输介质是数据从一台设备传到另一台设备的"路"。主要分为两大类：
- **有线 (Guided/Wired)**：铜缆、光纤
- **无线 (Unguided/Wireless)**：WiFi、微波、卫星

---

### 5.1 铜缆 (Copper Cable)

#### 双绞线 (Twisted Pair Cable)

最常见的网络线缆。两根铜线绞在一起，减少电磁干扰。

**种类：**
| 类型 | 带宽 | 最大速度 | 最大距离 | 用途 |
|------|------|---------|---------|------|
| Cat 5 | 100 MHz | 100 Mbps | 100m | 旧网络 |
| Cat 5e | 100 MHz | 1 Gbps | 100m | 家庭/办公 |
| Cat 6 | 250 MHz | 10 Gbps (55m) | 100m | 企业网络 |
| Cat 6a | 500 MHz | 10 Gbps | 100m | 数据中心 |

**UTP vs STP：**
- **UTP (Unshielded Twisted Pair)**：无屏蔽层，便宜但容易受干扰
- **STP (Shielded Twisted Pair)**：有金属屏蔽层，抗干扰但贵且硬

> **为什么要"绞"在一起？** 当两根线绞在一起时，外部的电磁干扰会同时影响两根线，在接收端可以通过差分信号抵消干扰。这就是"双绞"的科学原理！

#### 同轴电缆 (Coaxial Cable)
- 中心导体被绝缘层和屏蔽层包裹
- 曾用于早期以太网 (Bus 拓扑)
- 现在主要用于有线电视 (Cable TV) 和宽带接入

### 5.2 光纤 (Fibre Optic Cable)

光纤用 **光信号** 而不是电信号传输数据。

**结构：**
- **纤芯 (Core)**：极细的玻璃/塑料丝，光在里面传播
- **包层 (Cladding)**：折射率较低的材料，让光在纤芯内部发生 **全内反射 (Total Internal Reflection)**
- **外套 (Jacket)**：保护层

**全内反射原理：**
当光从密度较高的介质（纤芯）射向密度较低的介质（包层），如果入射角大于 **临界角 (Critical Angle)**，光就会完全反射回纤芯内部，不会泄漏出去。光就这样在纤芯中"弹来弹去"地前进。

**光纤的优势：**
- **速度极快**：可达 100 Gbps 甚至更高
- **传输距离远**：可达数十公里不需要中继器
- **不受电磁干扰 (EMI)**：因为传的是光不是电
- **安全性高**：很难被窃听（不会发射电磁辐射）
- **重量轻**

**光纤的劣势：**
- 成本高（材料和安装都贵）
- 脆弱，弯折可能损坏纤芯
- 需要专业设备和技术人员安装
- 接头连接复杂

**单模 vs 多模光纤：**
| 类型 | 纤芯直径 | 传输距离 | 成本 | 用途 |
|------|---------|---------|------|------|
| 单模 (Single-mode) | ~9 μm | 远 (>10km) | 高 | 长距离/WAN |
| 多模 (Multi-mode) | ~50 μm | 近 (<2km) | 低 | 短距离/LAN |

In [ ]:
# 可视化：光纤全内反射原理
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ── Fibre optic cross-section ──
ax = axes[0]
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.set_title('Fibre Optic Cable: Total Internal Reflection', fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')

# Cable layers
jacket = mpatches.FancyBboxPatch((1, 0.5), 12, 3, boxstyle="round,pad=0.1",
                                   facecolor='#424242', edgecolor='black', lw=2)
ax.add_patch(jacket)

cladding = mpatches.FancyBboxPatch((1.5, 0.8), 11, 2.4, boxstyle="round,pad=0.1",
                                     facecolor='#90CAF9', edgecolor='#1565C0', lw=1.5)
ax.add_patch(cladding)

core = mpatches.FancyBboxPatch((2, 1.3), 10, 1.4, boxstyle="round,pad=0.1",
                                 facecolor='#E3F2FD', edgecolor='#42A5F5', lw=1)
ax.add_patch(core)

# Light ray bouncing
light_x = [2.2, 3.5, 5.0, 6.5, 8.0, 9.5, 11.0, 11.8]
light_y = [2.0, 2.5, 1.5, 2.5, 1.5, 2.5, 1.5, 2.0]
ax.plot(light_x, light_y, 'r-', lw=2.5, alpha=0.9)
for x, y in zip(light_x, light_y):
    ax.plot(x, y, 'ro', markersize=4)

# Labels
ax.text(13.5, 3.2, 'Jacket (外套)', fontsize=9, color='white', fontweight='bold')
ax.text(13.5, 2.0, 'Cladding (包层)', fontsize=9, color='#1565C0')
ax.text(7, 2.0, 'Core (纤芯)', fontsize=10, color='#1565C0', ha='center', fontweight='bold')
ax.text(7, 0.15, 'Light bounces inside the core via Total Internal Reflection',
        ha='center', fontsize=10, style='italic', color='red')

# Light source arrow
ax.annotate('Light\nin', xy=(2.2, 2.0), xytext=(0.5, 2.0),
            fontsize=9, color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

# ── Speed comparison ──
ax2 = axes[1]
media_types = ['Cat 5\n(Copper)', 'Cat 5e\n(Copper)', 'Cat 6\n(Copper)',
               'Single-mode\n(Fibre)', 'Multi-mode\n(Fibre)']
speeds = [100, 1000, 10000, 100000, 10000]  # Mbps
colors_bar = ['#FF8A65', '#FF7043', '#F4511E', '#42A5F5', '#90CAF9']

bars = ax2.barh(media_types, speeds, color=colors_bar, edgecolor='black', linewidth=1)
ax2.set_xlabel('Maximum Speed (Mbps)', fontsize=12)
ax2.set_title('Transmission Media Speed Comparison', fontsize=14, fontweight='bold')
ax2.set_xscale('log')

for bar, speed in zip(bars, speeds):
    label = f'{speed} Mbps' if speed < 1000 else f'{speed/1000:.0f} Gbps'
    ax2.text(bar.get_width() * 1.2, bar.get_y() + bar.get_height()/2,
             label, va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('fibre_optic.png', dpi=100, bbox_inches='tight')
plt.show()

### 5.3 无线传输 (Wireless Transmission)

#### WiFi (无线局域网)

**WiFi** 使用 **无线电波 (Radio Waves)** 在短距离内传输数据。

| WiFi 标准 | 频率 | 最大速度 | 发布年份 |
|-----------|------|---------|---------|
| 802.11b | 2.4 GHz | 11 Mbps | 1999 |
| 802.11g | 2.4 GHz | 54 Mbps | 2003 |
| 802.11n (WiFi 4) | 2.4/5 GHz | 600 Mbps | 2009 |
| 802.11ac (WiFi 5) | 5 GHz | 6.9 Gbps | 2014 |
| 802.11ax (WiFi 6) | 2.4/5/6 GHz | 9.6 Gbps | 2020 |

**WiFi 的局限：**
- 信号受墙壁、金属等物体阻挡
- 距离越远信号越弱
- 容易受到其他无线设备干扰
- 安全性不如有线（信号在空气中传播可以被截获）

#### 微波 (Microwave)

- 使用高频无线电波（频率更高、波长更短）
- **必须直线传播 (Line of Sight)**——中间不能有障碍物
- 用于点对点的长距离通信
- 例子：手机基站之间的通信

#### 卫星通信 (Satellite Communication)

- 信号从地面站发送到太空中的卫星，卫星再转发到另一个地面站
- 可以覆盖极大的区域（甚至全球）
- **延迟高**：信号要往返太空 36,000km（地球同步轨道）
- 用于偏远地区通信、GPS、卫星电视

## 6. 有线 vs 无线网络对比

| 特性 | 有线网络 (Wired) | 无线网络 (Wireless) |
|------|----------------|-------------------|
| 速度 | 更快、更稳定 | 较慢、波动大 |
| 可靠性 | 高（不受干扰） | 受墙壁、干扰影响 |
| 安全性 | 高（需要物理接入） | 较低（信号可被截获） |
| 移动性 | 差（被线限制） | 好（自由移动） |
| 安装难度 | 需要布线 | 简单（不需要线缆） |
| 成本 | 线缆+安装费用 | 设备成本（WAP等） |
| 维护 | 线缆可能损坏 | 信号覆盖问题 |
| 适用场景 | 办公室、数据中心 | 移动设备、咖啡店 |

### 什么时候用有线？什么时候用无线？

**用有线的场景：**
- 需要高速稳定连接（服务器、游戏PC）
- 安全性要求高（银行、医院）
- 固定位置的设备（台式电脑）

**用无线的场景：**
- 需要移动性（手机、笔记本）
- 难以布线的位置（老建筑、临时场所）
- 大量访客需要临时连接（会议室、咖啡店）

In [ ]:
# 可视化：传输介质综合对比
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Radar chart: Media comparison ──
categories = ['Speed', 'Distance', 'Cost\n(lower=better)', 'Security', 'Ease of\nInstall']
N = len(categories)

# Values (1-5 scale)
media_data = {
    'Copper (Cat6)':   [3, 2, 4, 4, 4],
    'Fibre Optic':     [5, 5, 1, 5, 1],
    'WiFi':            [2, 1, 3, 2, 5],
    'Satellite':       [1, 5, 1, 3, 2],
}

angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

ax = axes[0]
ax = fig.add_subplot(121, polar=True)
ax.set_title('Transmission Media Comparison\n(1-5 scale)', fontsize=12, fontweight='bold', pad=20)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=8)
ax.set_ylim(0, 5)

colors_radar = ['#FF7043', '#42A5F5', '#66BB6A', '#AB47BC']
for (name, values), c in zip(media_data.items(), colors_radar):
    vals = values + values[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=name, color=c)
    ax.fill(angles, vals, alpha=0.1, color=c)

ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

# ── Bar chart: Max distance ──
ax2 = axes[1]
media_names = ['Cat 5e\n(Copper)', 'Cat 6\n(Copper)', 'Multi-mode\nFibre', 'Single-mode\nFibre',
               'WiFi\n(indoor)', 'Microwave', 'Satellite']
distances = [100, 100, 2000, 80000, 50, 50000, 36000000]  # meters
bar_colors = ['#FF8A65', '#F4511E', '#90CAF9', '#42A5F5', '#66BB6A', '#AB47BC', '#78909C']

bars = ax2.barh(media_names, distances, color=bar_colors, edgecolor='black', lw=1)
ax2.set_xlabel('Maximum Distance (meters) [log scale]', fontsize=10)
ax2.set_title('Maximum Transmission Distance', fontsize=12, fontweight='bold')
ax2.set_xscale('log')

for bar, d in zip(bars, distances):
    if d >= 1000:
        label = f'{d/1000:.0f} km'
    else:
        label = f'{d} m'
    ax2.text(bar.get_width() * 1.5, bar.get_y() + bar.get_height()/2,
             label, va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('media_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 6.1 数据传输速率 (Data Transmission Rates)

### 关键术语

| 术语 | 含义 | 单位 |
|------|------|------|
| **Bandwidth（带宽）** | 传输介质能承载的最大数据量 | bps, Mbps, Gbps |
| **Throughput（吞吐量）** | 实际传输的数据量（通常 < 带宽） | bps, Mbps, Gbps |
| **Latency（延迟）** | 数据从源到目的地所需的时间 | ms (毫秒) |

### 单位换算

```
1 Kbps = 1,000 bps        (千比特每秒)
1 Mbps = 1,000,000 bps    (兆比特每秒)
1 Gbps = 1,000,000,000 bps (吉比特每秒)
```

> **注意**：bps 是 **bits** per second（比特），而不是 **bytes**！
> 1 byte = 8 bits，所以 100 Mbps 的下载速度大约是 12.5 MB/s。

### 影响数据传输速率的因素

1. **传输介质**：光纤 > 铜缆 > 无线
2. **距离**：距离越远，信号衰减越严重
3. **干扰 (Noise)**：电磁干扰会降低信号质量
4. **网络拥塞**：用户越多，每人分到的带宽越少
5. **硬件性能**：网卡、Switch、Router 的处理能力

In [ ]:
# 数据传输计算练习
print("=" * 55)
print("  Data Transfer Rate Calculations")
print("=" * 55)

# 问题 1: 下载时间
file_size_mb = 700  # MB (一部电影)
bandwidth_mbps = 100  # Mbps

file_size_bits = file_size_mb * 8  # Convert MB to Mb
time_seconds = file_size_bits / bandwidth_mbps

print(f"\nQ: How long to download a {file_size_mb} MB file at {bandwidth_mbps} Mbps?")
print(f"   File size = {file_size_mb} MB = {file_size_mb} x 8 = {file_size_bits} Mb")
print(f"   Time = {file_size_bits} Mb / {bandwidth_mbps} Mbps = {time_seconds:.1f} seconds")
print(f"        = {time_seconds/60:.1f} minutes")

# 问题 2: 不同介质的对比
print(f"\n{'=' * 55}")
print(f"  Download a 4 GB movie on different connections:")
print(f"{'=' * 55}")
file_gb = 4
file_mb = file_gb * 1024
connections = [
    ("Dial-up (56 Kbps)", 0.056),
    ("ADSL (24 Mbps)", 24),
    ("Cable (100 Mbps)", 100),
    ("Fibre (1 Gbps)", 1000),
    ("5G (1 Gbps)", 1000),
]
for name, speed_mbps in connections:
    time_s = (file_mb * 8) / speed_mbps
    if time_s > 3600:
        time_str = f"{time_s/3600:.1f} hours"
    elif time_s > 60:
        time_str = f"{time_s/60:.1f} minutes"
    else:
        time_str = f"{time_s:.1f} seconds"
    print(f"  {name:<25} -> {time_str}")

## 7. 练习题 (Practice Exercises)

### 选择题

**Q1.** NIC 的主要功能是什么？
- A) 放大网络信号
- B) 将数据转换为可以在网络上传输的信号
- C) 连接不同的网络
- D) 过滤网络流量

**Q2.** Switch 和 Hub 的关键区别是什么？
- A) Switch 更便宜
- B) Hub 可以连接更多设备
- C) Switch 只将数据发送到目标端口，Hub 广播到所有端口
- D) Hub 比 Switch 更安全

**Q3.** Router 使用什么地址来转发数据？
- A) MAC 地址
- B) IP 地址
- C) 端口号
- D) 域名

**Q4.** 光纤相比铜缆的主要优势不包括：
- A) 传输速度更快
- B) 不受电磁干扰
- C) 安装更容易
- D) 传输距离更远

**Q5.** 以下哪种情况最适合使用无线网络？
- A) 银行的核心服务器连接
- B) 咖啡店为顾客提供上网服务
- C) 数据中心的服务器之间通信
- D) 工厂的固定监控摄像头

---

### 参考答案
1. **B** — NIC 将数字数据转换为网络信号
2. **C** — 这是 Switch 和 Hub 最本质的区别
3. **B** — Router 工作在网络层，使用 IP 地址
4. **C** — 光纤安装困难是它的主要缺点
5. **B** — 咖啡店顾客需要移动性和临时连接

### 简答题

**Q6.** 解释为什么光纤中要使用"全内反射"原理。如果没有全内反射会怎样？（3分）

<details>
<summary>点击查看参考答案</summary>

- 全内反射让光在纤芯内部不断反射前进，不会泄漏到外面（1分）
- 这保证了光信号能够传输很长距离而不损失太多能量（1分）
- 如果没有全内反射，光会折射出纤芯进入包层并消失，信号将迅速衰减无法传输（1分）

</details>

**Q7.** 一所学校正在建设新的计算机网络。解释为什么他们应该选择 Switch 而不是 Hub。（4分）

<details>
<summary>点击查看参考答案</summary>

- Switch 只将数据发送到目标设备，不会像 Hub 那样广播给所有人，节省带宽（1分）
- 每个端口有独立的带宽，不会因为其他设备在通信而减速（1分）
- 更安全：数据不会被非目标设备看到（1分）
- 减少数据碰撞，提高网络效率和可靠性（1分）

</details>

---
*本节结束！下一节我们将学习互联网与万维网。*